# Modelado con Tuning y Thresholds

Notebook para revisar la segunda ronda de modelado enfocada en recall de la clase pobre.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

In [ ]:
from utils.paths import build_paths
report_path = PROJECT_ROOT / 'reports' / 'scenario_b_modeling_recall' / 'model_recall_optimization_summary.csv'

paths = build_paths(PROJECT_ROOT)
dataset, top20_df = load_inputs(paths)
top30_full, _, missing = detect_feature_sets(dataset, top20_df)

print('Dataset:', dataset.shape)
print('Top30 full:', len(top30_full))
print('Missing:', missing)

In [ ]:
summary = pd.read_csv(report_path).sort_values(['recall', 'f1', 'precision'], ascending=[False, False, False]).reset_index(drop=True)
summary.head(10)

In [ ]:
best_recall = summary.iloc[0]
best_balanced = summary.sort_values(['f1', 'recall', 'precision'], ascending=[False, False, False]).iloc[0]
best_precise = summary.sort_values(['precision', 'f1', 'recall'], ascending=[False, False, False]).iloc[0]

print('Mejor por recall:')
print(best_recall)
print('\nMejor balanceado:')
print(best_balanced)
print('\nMejor por precision:')
print(best_precise)

In [ ]:
def confusion_df(row):
    import json
    cm = json.loads(row['confusion_matrix']) if isinstance(row['confusion_matrix'], str) else row['confusion_matrix']
    return pd.DataFrame(cm, index=['Real: no pobre', 'Real: pobre'], columns=['Pred: no pobre', 'Pred: pobre'])

confusion_df(best_recall)

In [ ]:
confusion_df(best_balanced)

In [ ]:
summary.groupby(['escenario', 'modelo_familia'])[['recall', 'precision', 'f1', 'pr_auc']].max().sort_values(['recall', 'f1'], ascending=[False, False])